In [60]:
import pandas as pd

df = pd.read_parquet("data/prepared_recipes.parquet")

df.head()

,name,ingredients,description,steps
0,roast butternut squash with maple syrup and gi...,"[butternut squash, ghee, maple syrup, pine nut...","found in times2, made it several times since w...","[preheat the oven to 350f, peel the squash , c..."
1,blond brownies with br sugar frosting,"[granulated sugar, vanilla, butter, eggs, flou...",have not tried these yet..but want to some day...,"[heat oven to 350 --, beat sugars , butter , v..."
2,hershey s double chocolate and peanut butter c...,"[butter, vanilla, cocoa, salt, nuts, sugar, eg...",chocolate cookies with chocolate and peanut bu...,"[preheat oven to 350 degrees, in a large mixin..."
3,latte frozen yogurt,"[sugar, cornstarch, low-fat milk, instant coff...",the flavor on this is absolutely amazing! mor...,"[in a 2-quart saucepan , combine sugar , insta..."
4,cinnamon quick bread,"[vegetable oil, egg, salt, cinnamon, sugar, bu...",yummy and easy! good with a cup of tea in the...,"[filling: mix and set aside, mix flour , bakin..."


In [61]:
len(df)

100000

In [62]:
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
tqdm.pandas(desc="Embedding recipes")

model = AutoModel.from_pretrained("google/bert_uncased_L-2_H-128_A-2")
tokenizer = AutoTokenizer.from_pretrained("google/bert_uncased_L-2_H-128_A-2")

def embed(text):
	if not isinstance(text, str):
		return None
	inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
	outputs = model(**inputs)
	return outputs.pooler_output.detach().numpy()


In [64]:
vector_limit = 1000

vector_df = pd.DataFrame(columns=["text", "vector"])

vector_df["text"] = (df["name"] + "\n" + df["ingredients"].apply(", ".join))[:vector_limit]
vector_df["vectors"] = vector_df["text"].progress_apply(embed)

vector_df = vector_df.dropna(subset=["vectors"])

Embedding recipes: 100%|██████████| 1000/1000 [00:39<00:00, 25.12it/s]


In [65]:
from vicinity import Vicinity, Backend, Metric
vicinity = Vicinity.from_vectors_and_items(
    vectors=vector_df["vectors"].tolist(),
    items=vector_df["text"].tolist(),
    backend_type=Backend.BASIC,
    metric=Metric.COSINE
)

In [66]:
vicinity.save('data/vicinity_recipe_vectors',overwrite=True)